# TWSE PPO + SMC 訓練與驗證流程

這份 notebook 專注在目前正式版 pipeline。

流程目標：

1. 確認環境與 GPU。
2. 下載訓練資料。
3. 建立 SMC 數值化特徵。
4. 建立 Gymnasium 虛擬市場。
5. 訓練 PPO。
6. 下載驗證資料。
7. 建立驗證用 SMC 特徵。
8. 輸出驗證報表與圖表。

目前模型：

- Model 1：`0050.TW vs 2330.TW`
- Model 2：`0050.TW vs 2330.TW vs 2412.TW`


## 1. 環境確定

Kernel crash 通常是底層套件匯入失敗造成，例如 PyTorch、CUDA DLL、NVIDIA driver 或 Stable-Baselines3 相依套件。

因此這裡拆成幾個小步驟：

- 1.1 先確認 Jupyter kernel 本身可執行。
- 1.2 再確認 PyTorch 與 CUDA。
- 1.3 再匯入一般資料與 RL 套件。
- 1.4 最後匯入本專案模組。

如果 kernel 在某一格 crash，就代表問題集中在那一格的套件。


### 1.1 Kernel 基本檢查

這一格不匯入 GPU 或大型套件，只確認目前 Jupyter kernel 可以正常跑 Python。


In [ ]:
import sys
from pathlib import Path

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")


### 1.2 PyTorch 與裝置檢查

這一格會檢查目前是否可以使用 GPU。

- 如果 CUDA 可用，使用 `cuda:0`。
- 如果 CUDA 不可用，自動改用 `cpu`。
- 最後會顯示目前實際使用的裝置。

注意：使用 CPU 可以讓 notebook 繼續跑，但 PPO 訓練速度會明顯變慢。


In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda:0"
    device_name = torch.cuda.get_device_name(0)
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = "cpu"
    device_name = "CPU"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Using device: {DEVICE}")
print(f"Device name: {device_name}")


### 1.3 套件匯入檢查

這一格確認資料處理、繪圖與 Stable-Baselines3 可以正常匯入。


In [ ]:
import gymnasium as gym
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

print(f"Gymnasium: {gym.__version__}")
print(f"Pandas: {pd.__version__}")
print("Stable-Baselines3 PPO import: OK")


### 1.4 Pipeline 共用邏輯說明

原本這一格是從 `twse_pipeline_common.py` 匯入共用模組。為了讓 notebook 更容易閱讀，現在把核心邏輯放回 notebook。

這一段主要控制：

- **模型設定**：定義要訓練哪些模型、每個模型使用哪些 ticker、模型存到哪裡。
- **市場限制**：台股手續費、交易稅、10% 漲停限制。
- **資料下載**：透過 yfinance 下載 OHLCV。
- **SMC 特徵**：計算 `PD_Pos`、`OB_Dist`、`FVG_Signal`、`Spread_ZScore`。
- **防止未來資訊**：所有 SMC 與 spread 特徵都延遲一日，避免同一天 K 棒資訊被拿來做同一天交易決策。
- **Gymnasium 虛擬市場**：控制 observation、action、交易執行、reward、資產淨值與交易紀錄。
- **績效報表**：計算 cumulative return、Sharpe ratio、max drawdown。


### 1.4.1 模型與市場參數

這一格定義模型配置與台股交易成本。


In [ ]:
import os
from dataclasses import dataclass

import numpy as np
import yfinance as yf
from gymnasium import spaces


BROKERAGE_FEE = 0.001425 * 0.6
PRICE_LIMIT_UP = 1.10
SHARPE_WINDOW = 20
EPS = 1e-8


@dataclass(frozen=True)
class ModelConfig:
    name: str
    tickers: tuple[str, ...]
    model_path: str
    mode: str


PAIR_CONFIG = ModelConfig(
    name="pair_0050_2330",
    tickers=("0050.TW", "2330.TW"),
    model_path="model/saved/ppo_model_pair.zip",
    mode="pair",
)

BASKET_CONFIG = ModelConfig(
    name="basket_0050_2330_2412",
    tickers=("0050.TW", "2330.TW", "2412.TW"),
    model_path="model/saved/ppo_model_basket.zip",
    mode="basket",
)

SELL_TAX = {
    "0050.TW": 0.001,
    "2330.TW": 0.003,
    "2412.TW": 0.003,
}

print("Model and market configs are ready.")


### 1.4.2 資料下載與 SMC 特徵

這一格控制原始資料下載與特徵工程。

重點是 `build_feature_frame()` 最後會把 SMC / spread 特徵 `shift(1)`，避免未來資訊。


In [ ]:
def download_ohlcv(tickers, start_date, end_date):
    frames = {}
    for ticker in tickers:
        df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)
        if df.empty:
            raise ValueError(f"No data downloaded for {ticker}")
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.reset_index()
        df = df.rename(
            columns={
                "Date": "date",
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close",
                "Adj Close": "adj_close",
                "Volume": "volume",
            }
        )
        required = ["date", "open", "high", "low", "close", "volume"]
        frames[ticker] = df[required].dropna().reset_index(drop=True)
    return frames


def add_smc_features(df):
    out = df.copy()

    high_20 = out["high"].rolling(20).max()
    low_20 = out["low"].rolling(20).min()
    dealing_range = (high_20 - low_20).replace(0, np.nan)
    out["PD_Pos"] = ((out["close"] - low_20) / dealing_range).clip(0.0, 1.0)

    ret_3d = out["close"].pct_change(3)
    bearish_candle = out["close"] < out["open"]
    ob_level = np.nan
    ob_levels = []
    for i in range(len(out)):
        if i >= 3 and ret_3d.iloc[i] > 0.03:
            start = max(0, i - 20)
            for j in range(i - 3, start - 1, -1):
                if bearish_candle.iloc[j]:
                    ob_level = out["high"].iloc[j]
                    break
        ob_levels.append(ob_level)
    out["ob_level"] = pd.Series(ob_levels, index=out.index).ffill()
    out["OB_Dist"] = ((out["close"] - out["ob_level"]) / out["close"]).replace([np.inf, -np.inf], np.nan)

    bullish_fvg = out["low"] > out["high"].shift(2)
    bearish_fvg = out["high"] < out["low"].shift(2)
    bull_low = out["high"].shift(2)
    bull_high = out["low"]
    bear_low = out["high"]
    bear_high = out["low"].shift(2)

    unfilled = np.zeros(len(out), dtype=np.float32)
    active_gaps = []
    for i in range(len(out)):
        still_active = []
        for gap in active_gaps:
            touches_gap = out["low"].iloc[i] <= gap["high"] and out["high"].iloc[i] >= gap["low"]
            if not touches_gap:
                still_active.append(gap)
        active_gaps = still_active

        if bullish_fvg.iloc[i]:
            active_gaps.append({"low": bull_low.iloc[i], "high": bull_high.iloc[i], "direction": "bull"})
        if bearish_fvg.iloc[i]:
            active_gaps.append({"low": bear_low.iloc[i], "high": bear_high.iloc[i], "direction": "bear"})

        unfilled[i] = 1.0 if active_gaps else 0.0
    out["FVG_Signal"] = unfilled

    out[["PD_Pos", "OB_Dist", "FVG_Signal"]] = out[["PD_Pos", "OB_Dist", "FVG_Signal"]].fillna(
        {"PD_Pos": 0.5, "OB_Dist": 0.0, "FVG_Signal": 0.0}
    )
    return out


def build_feature_frame(raw_frames, config):
    feature_frames = []
    for ticker in config.tickers:
        df = add_smc_features(raw_frames[ticker])
        prefix = ticker.replace(".", "_")
        cols = ["date", "open", "high", "low", "close", "volume", "PD_Pos", "OB_Dist", "FVG_Signal"]
        renamed = df[cols].rename(columns={col: f"{prefix}_{col}" for col in cols if col != "date"})
        feature_frames.append(renamed)

    merged = feature_frames[0]
    for frame in feature_frames[1:]:
        merged = pd.merge(merged, frame, on="date", how="inner")

    if config.mode in {"pair", "basket"}:
        pairs = (
            [(config.tickers[0], config.tickers[1])]
            if config.mode == "pair"
            else [
                (config.tickers[i], config.tickers[j])
                for i in range(len(config.tickers))
                for j in range(i + 1, len(config.tickers))
            ]
        )
        for left_ticker, right_ticker in pairs:
            left = left_ticker.replace(".", "_")
            right = right_ticker.replace(".", "_")
            col = f"Spread_ZScore_{left}_{right}"
            spread = np.log(merged[f"{left}_close"]) - np.log(merged[f"{right}_close"])
            mean = spread.rolling(20).mean()
            std = spread.rolling(20).std().replace(0, np.nan)
            merged[col] = ((spread - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if config.mode == "pair":
        left = config.tickers[0].replace(".", "_")
        right = config.tickers[1].replace(".", "_")
        merged["Spread_ZScore"] = merged[f"Spread_ZScore_{left}_{right}"]
    else:
        merged["Spread_ZScore"] = 0.0

    feature_cols = [
        col
        for col in merged.columns
        if any(token in col for token in ("PD_Pos", "OB_Dist", "FVG_Signal", "Spread_ZScore"))
    ]
    merged[feature_cols] = merged[feature_cols].shift(1)

    return merged.dropna().reset_index(drop=True)


print("Data download and SMC feature functions are ready.")


### 1.4.3 Gymnasium 虛擬市場與績效計算

這一格控制交易環境。

模型每一步會看到：

- SMC / spread 特徵。
- 目前各標的持倉權重。
- 現金比例。

模型輸出 action 後，環境會計算目標權重、交易成本、交易稅、回撤、reward 與資產淨值。


In [ ]:
class TWTradingEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, feature_df, config, initial_balance=1_000_000.0, max_position=0.95):
        super().__init__()
        self.df = feature_df.reset_index(drop=True)
        self.config = config
        self.initial_balance = float(initial_balance)
        self.max_position = float(max_position)
        self.asset_prefixes = [ticker.replace(".", "_") for ticker in config.tickers]
        self.feature_cols = self._make_feature_cols()

        obs_dim = len(self.feature_cols) + len(self.config.tickers) + 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        action_dim = len(self.config.tickers) if self.config.mode == "basket" else 1
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(action_dim,), dtype=np.float32)

    def _make_feature_cols(self):
        cols = []
        for prefix in self.asset_prefixes:
            cols.extend([f"{prefix}_PD_Pos", f"{prefix}_OB_Dist", f"{prefix}_FVG_Signal"])
        if self.config.mode == "pair":
            cols.append("Spread_ZScore")
        elif self.config.mode == "basket":
            cols.extend([col for col in self.df.columns if col.startswith("Spread_ZScore_")])
        return cols

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.cash = self.initial_balance
        self.shares = np.zeros(len(self.config.tickers), dtype=np.float64)
        self.net_worth = self.initial_balance
        self.max_net_worth = self.initial_balance
        self.prev_drawdown = 0.0
        self.portfolio_returns = []
        self.history = []
        return self._get_obs(), {}

    def _prices(self, step=None):
        idx = self.current_step if step is None else step
        return np.array([self.df.loc[idx, f"{prefix}_close"] for prefix in self.asset_prefixes], dtype=np.float64)

    def _prev_closes(self):
        prev_step = max(0, self.current_step - 1)
        return np.array([self.df.loc[prev_step, f"{prefix}_close"] for prefix in self.asset_prefixes], dtype=np.float64)

    def _get_weights(self, prices=None, net_worth=None):
        prices = self._prices() if prices is None else prices
        net_worth = self.net_worth if net_worth is None else net_worth
        if net_worth <= 0:
            return np.zeros(len(self.config.tickers), dtype=np.float64)
        return (self.shares * prices) / net_worth

    def _get_obs(self):
        market_features = self.df.loc[self.current_step, self.feature_cols].to_numpy(dtype=np.float32)
        prices = self._prices()
        net_worth = self.cash + float(np.dot(self.shares, prices))
        weights = self._get_weights(prices, net_worth).astype(np.float32)
        cash_ratio = np.array([self.cash / max(net_worth, EPS)], dtype=np.float32)
        return np.concatenate([market_features, weights, cash_ratio]).astype(np.float32)

    def _target_weights(self, raw_action):
        action_array = np.asarray(raw_action, dtype=np.float64).reshape(-1)
        if self.config.mode == "pair":
            action = float(np.clip(action_array[0], -1.0, 1.0))
            weights = np.zeros(2, dtype=np.float64)
            if action >= 0:
                weights[0] = action * self.max_position
            else:
                weights[1] = -action * self.max_position
            return weights

        if self.config.mode == "basket":
            scores = np.clip(action_array[: len(self.config.tickers)], 0.0, 1.0)
            if scores.sum() <= EPS:
                return np.zeros(len(self.config.tickers), dtype=np.float64)
            return (scores / scores.sum()) * self.max_position

        action = float(np.clip(action_array[0], -1.0, 1.0))
        return np.array([max(0.0, action) * self.max_position], dtype=np.float64)

    def _execute_rebalance(self, target_weights, prices):
        prev_prices = self._prev_closes()
        net_worth_before_trade = self.cash + float(np.dot(self.shares, prices))
        current_values = self.shares * prices
        target_values = target_weights * net_worth_before_trade
        value_diffs = target_values - current_values
        transaction_cost = 0.0
        failed_orders = 0

        for idx, value_diff in enumerate(value_diffs):
            ticker = self.config.tickers[idx]
            price = prices[idx]
            limit_up = prev_prices[idx] * PRICE_LIMIT_UP

            if abs(value_diff) > EPS and price >= limit_up:
                failed_orders += 1
                continue

            if value_diff > 0:
                buy_value = min(value_diff, max(0.0, self.cash / (1.0 + BROKERAGE_FEE)))
                fee = buy_value * BROKERAGE_FEE
                self.cash -= buy_value + fee
                self.shares[idx] += buy_value / price
                transaction_cost += fee
            elif value_diff < 0:
                sell_value = min(-value_diff, self.shares[idx] * price)
                fee = sell_value * BROKERAGE_FEE
                tax = sell_value * SELL_TAX[ticker]
                self.cash += sell_value - fee - tax
                self.shares[idx] -= sell_value / price
                transaction_cost += fee + tax

        return transaction_cost, failed_orders

    def step(self, action):
        raw_action_array = np.asarray(action, dtype=np.float64).reshape(-1)
        raw_action = float(raw_action_array[0])
        prices = self._prices()
        prev_net_worth = self.net_worth
        target_weights = self._target_weights(raw_action_array)
        transaction_cost, failed_orders = self._execute_rebalance(target_weights, prices)

        self.net_worth = self.cash + float(np.dot(self.shares, prices))
        pnl = self.net_worth - prev_net_worth
        pnl_rate = pnl / max(prev_net_worth, EPS)
        self.portfolio_returns.append(pnl_rate)

        recent = np.array(self.portfolio_returns[-SHARPE_WINDOW:], dtype=np.float64)
        if len(recent) > 1 and np.std(recent) > EPS:
            sharpe_adjustment = max(0.0, np.mean(recent) / (np.std(recent) + EPS) * np.sqrt(252))
        else:
            sharpe_adjustment = 1.0

        self.max_net_worth = max(self.max_net_worth, self.net_worth)
        drawdown = (self.max_net_worth - self.net_worth) / max(self.max_net_worth, EPS)
        drawdown_penalty = max(0.0, drawdown - self.prev_drawdown)
        self.prev_drawdown = drawdown

        cost_rate = transaction_cost / max(prev_net_worth, EPS)
        reward = (pnl_rate * sharpe_adjustment) - cost_rate - (2.0 * drawdown_penalty)

        weights = self._get_weights(prices, self.net_worth)
        self.history.append(
            {
                "date": self.df.loc[self.current_step, "date"],
                "raw_action": raw_action,
                "net_worth": self.net_worth,
                "pnl": pnl,
                "pnl_rate": pnl_rate,
                "transaction_cost": transaction_cost,
                "drawdown": drawdown,
                "reward": reward,
                "failed_orders": failed_orders,
                **{f"raw_action_{ticker}": raw_action_array[i] for i, ticker in enumerate(self.config.tickers) if i < len(raw_action_array)},
                **{f"weight_{ticker}": weights[i] for i, ticker in enumerate(self.config.tickers)},
            }
        )

        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        truncated = False
        obs = np.zeros(self.observation_space.shape, dtype=np.float32) if terminated else self._get_obs()
        info = {"net_worth": self.net_worth, "drawdown": drawdown, "transaction_cost": transaction_cost}
        return obs, float(reward), terminated, truncated, info


def calculate_metrics(history_df, initial_balance=1_000_000.0):
    if history_df.empty:
        return {"cumulative_return": 0.0, "sharpe_ratio": 0.0, "max_drawdown": 0.0}
    net_worth = history_df["net_worth"].astype(float)
    cumulative_return = (net_worth.iloc[-1] / initial_balance) - 1.0
    returns = net_worth.pct_change().dropna()
    sharpe = 0.0
    if len(returns) > 1 and returns.std() > EPS:
        sharpe = (returns.mean() / returns.std()) * np.sqrt(252)
    roll_max = net_worth.cummax()
    drawdown = (net_worth - roll_max) / roll_max
    return {
        "cumulative_return": float(cumulative_return),
        "sharpe_ratio": float(sharpe),
        "max_drawdown": float(drawdown.min()),
    }


def ensure_model_dir(path):
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)


print("Gymnasium environment and metric functions are ready.")


## 2. 資料下載

這一步只下載原始 OHLCV 資料。

先不做特徵，也不建立環境，讓資料來源與時間區間保持清楚。


## 2.1 訓練資料的時間區間選擇

設定 in-sample 訓練區間與訓練參數。


In [ ]:
TRAIN_START = "2018-01-01"
TRAIN_END = "2024-12-31"
TOTAL_TIMESTEPS = 500_000
INITIAL_BALANCE = 1_000_000.0

MODEL_CONFIGS = [PAIR_CONFIG, BASKET_CONFIG]

print(f"Training period: {TRAIN_START} to {TRAIN_END}")
for config in MODEL_CONFIGS:
    print(f"{config.name}: {', '.join(config.tickers)} -> {config.model_path}")


In [ ]:
train_raw_frames = {}

for config in MODEL_CONFIGS:
    print(f"Downloading training data for {config.name}: {config.tickers}")
    train_raw_frames[config.name] = download_ohlcv(config.tickers, TRAIN_START, TRAIN_END)

for model_name, frames in train_raw_frames.items():
    print(f"\n{model_name}")
    for ticker, df in frames.items():
        print(f"  {ticker}: {df['date'].min()} -> {df['date'].max()}, rows={len(df)}")
        display(df.head())


## 3. SMC 資料正規化

這裡把 OHLCV 轉成 PPO 可讀的數值特徵。

目前的 SMC 特徵已經是模型可直接使用的尺度：

- `PD_Pos`：20 日 dealing range 位置，約 `0` 到 `1`。
- `OB_Dist`：距離 order block 的百分比距離。
- `FVG_Signal`：是否存在尚未回補 FVG，值為 `0` 或 `1`。
- `Spread_ZScore`：pair / basket 的 rolling z-score。

注意：這裡不額外做 mean/std scaling，避免 notebook 訓練與 `app.py`、`predict_pipeline.py` 推論輸入尺度不一致。

防止未來資訊：`build_feature_frame()` 會把所有 SMC 與 spread 特徵延遲一日。也就是第 t 天決策時，只能使用第 t-1 天以前已知的特徵。


In [ ]:
def smc_feature_columns(df):
    tokens = ["PD_Pos", "OB_Dist", "FVG_Signal", "Spread_ZScore"]
    return [col for col in df.columns if any(token in col for token in tokens)]


train_feature_frames = {}

for config in MODEL_CONFIGS:
    feature_df = build_feature_frame(train_raw_frames[config.name], config)
    train_feature_frames[config.name] = feature_df

    feature_cols = smc_feature_columns(feature_df)
    print(f"\n{config.name}")
    print(f"rows={len(feature_df)}, smc_feature_count={len(feature_cols)}")
    display(feature_df[["date"] + feature_cols].head())


## 4. `gymnasium` 虛擬市場環境建立

把 SMC 特徵放進 `TWTradingEnv`。

環境負責：

- 現金與持倉。
- 買賣手續費。
- 交易稅。
- 10% 漲停限制。
- reward、drawdown 與交易紀錄。


In [ ]:
def make_vec_env(config, feature_df):
    return DummyVecEnv([
        lambda: TWTradingEnv(
            feature_df=feature_df,
            config=config,
            initial_balance=INITIAL_BALANCE,
        )
    ])


train_envs = {}

for config in MODEL_CONFIGS:
    env = make_vec_env(config, train_feature_frames[config.name])
    train_envs[config.name] = env
    raw_env = env.envs[0]
    print(
        f"{config.name}: obs_shape={raw_env.observation_space.shape}, "
        f"action_shape={raw_env.action_space.shape}"
    )


## 5. PPO 訓練

開始訓練 PPO agent。

訓練完成後，模型會儲存在 `model/saved/`。


In [ ]:
def train_ppo_model(config, env):
    policy_kwargs = dict(net_arch=dict(pi=[128, 128], vf=[128, 128]))
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=2e-4,
        n_steps=2048,
        batch_size=128,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.005,
        target_kl=0.03,
        policy_kwargs=policy_kwargs,
        verbose=1,
        device=DEVICE,
    )
    model.learn(total_timesteps=TOTAL_TIMESTEPS)
    ensure_model_dir(config.model_path)
    model.save(config.model_path)
    print(f"Saved model: {config.model_path}")
    return model


trained_models = {}

for config in MODEL_CONFIGS:
    print(f"\n=== Training {config.name} ===")
    trained_models[config.name] = train_ppo_model(config, train_envs[config.name])


## 6. 驗證資料的時間區間選擇

設定 out-of-sample 驗證區間。

這段資料只用來驗證，不會更新 PPO 權重。


In [ ]:
VAL_START = "2025-01-01"
VAL_END = "2026-05-01"

print(f"Validation period: {VAL_START} to {VAL_END}")


## 7. SMC 正規化

驗證資料使用與訓練資料相同的特徵生成邏輯。

這一步只建立 out-of-sample SMC 特徵，不會重新訓練，也不會修改模型權重。


In [ ]:
val_raw_frames = {}
val_feature_frames = {}

for config in MODEL_CONFIGS:
    print(f"Downloading validation data for {config.name}: {config.tickers}")
    raw_frames = download_ohlcv(config.tickers, VAL_START, VAL_END)
    feature_df = build_feature_frame(raw_frames, config)

    val_raw_frames[config.name] = raw_frames
    val_feature_frames[config.name] = feature_df

    feature_cols = smc_feature_columns(feature_df)
    print(f"\n{config.name}: rows={len(feature_df)}, smc_feature_count={len(feature_cols)}")
    display(feature_df[["date"] + feature_cols].head())


## 8. 驗證資料報表輸出

載入訓練完成的模型，逐日執行：

```python
model.predict(obs, deterministic=True)
```

驗證階段不會呼叫 `model.learn()`，因此不會更新 policy weights。


In [ ]:
def run_validation(config, feature_df):
    model = PPO.load(config.model_path, device=DEVICE)
    env = TWTradingEnv(
        feature_df=feature_df,
        config=config,
        initial_balance=INITIAL_BALANCE,
    )

    obs, _ = env.reset()
    done = False
    action_logs = []

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        action_values = action.reshape(-1).astype(float)
        row = {"date": env.df.loc[env.current_step, "date"], "action": float(action_values[0])}

        for idx, ticker in enumerate(config.tickers):
            if idx < len(action_values):
                row[f"action_{ticker}"] = float(action_values[idx])
        action_logs.append(row)

        obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    history = pd.DataFrame(env.history)
    actions = pd.DataFrame(action_logs)
    metrics = calculate_metrics(history, initial_balance=INITIAL_BALANCE)
    return history, actions, metrics


validation_results = []

for config in MODEL_CONFIGS:
    print(f"\n=== Validating {config.name} ===")
    history, actions, metrics = run_validation(config, val_feature_frames[config.name])
    validation_results.append({
        "config": config,
        "history": history,
        "actions": actions,
        "metrics": metrics,
    })

summary = pd.DataFrame([
    {
        "model": result["config"].name,
        "tickers": ", ".join(result["config"].tickers),
        "cumulative_return": result["metrics"]["cumulative_return"],
        "sharpe_ratio": result["metrics"]["sharpe_ratio"],
        "max_drawdown": result["metrics"]["max_drawdown"],
    }
    for result in validation_results
])

display(summary.style.format({
    "cumulative_return": "{:.2%}",
    "sharpe_ratio": "{:.2f}",
    "max_drawdown": "{:.2%}",
}))


In [ ]:
for result in validation_results:
    config = result["config"]
    history = result["history"]
    actions = result["actions"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=history["date"],
        y=history["net_worth"],
        mode="lines",
        name="Net Worth",
    ))
    fig.update_layout(
        title=f"{config.name} Validation Net Worth",
        yaxis_title="NTD",
        template="plotly_dark",
    )
    fig.show()

    action_cols = [col for col in actions.columns if col.startswith("action_")]
    fig_action = go.Figure()

    if action_cols:
        for col in action_cols:
            fig_action.add_trace(go.Scatter(
                x=actions["date"],
                y=actions[col],
                mode="lines",
                name=col.replace("action_", ""),
            ))
    else:
        fig_action.add_trace(go.Scatter(
            x=actions["date"],
            y=actions["action"],
            mode="lines",
            name="action",
        ))

    fig_action.update_layout(
        title=f"{config.name} Deterministic Actions",
        yaxis_title="Action [-1, 1]",
        template="plotly_dark",
    )
    fig_action.show()
